In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

In [2]:
data = pd.read_parquet('../output/data_with_embeddings.parquet')

In [3]:
data

,book_id,description,summaries,preprocessed_summaries,embeddings
0,1,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,Sixteen-year-old Katniss Everdeen regards it a...,Sixteen-year-old Katniss Everdeen regards it a...,"[-0.07609928399324417, -0.08204061537981033, 0..."
1,2,Harry Potter's life is miserable. His parents ...,Harry Potter is the only person to have ever s...,Harry Potter is the only person to have ever s...,"[0.05773061513900757, -0.007510815281420946, 0..."
2,3,About three things I was absolutely positive.\...,"""I was unconditionally and irrevocably in love...","""I was unconditionally and irrevocably in love...","[0.03286465257406235, -0.04374625161290169, -0..."
3,4,The unforgettable novel of a childhood in a sl...,To Kill A Mockingbird is regarded as a masterp...,To Kill A Mockingbird is regarded as a masterp...,"[0.048103220760822296, 0.00975493248552084, -0..."
4,5,Alternate Cover Edition ISBN: 0743273567 (ISBN...,F. Scott Fitzgerald's third book stands as the...,F. Scott Fitzgerald's third book stands as the...,"[-0.025573400780558586, -0.025715366005897522,..."
...,...,...,...,...,...
8934,9981,"A high-school girl in Harlem, Geneva Settle, i...","High-school girl in Harlem, Geneva Settle, is ...","High-school girl in Harlem, Geneva Settle, is ...","[-0.016489796340465546, -0.06769339740276337, ..."
8935,9982,In Karen Marie Moning’s latest installment of ...,Karen Marie Moning’s latest installment of the...,Karen Marie Moning’s latest installment of the...,"[-0.02939021587371826, -0.058736350387334824, ..."
8936,9985,"In the year 2000, computers are the new superp...","In the year 2000, computers are the new superp...","In the year 2000, computers are the new superp...","[-0.054065655916929245, -0.04327474534511566, ..."
8937,9987,A CIA agent's two-year-old child was stolen in...,Catherine Ling's two-year-old child was stolen...,Catherine Ling's two-year-old child was stolen...,"[-0.05844102054834366, -0.03694569692015648, 0..."


### Data cleaning

We have performed following preprocessing steps:
- white spaces cleaning
- non-alpha characters removal (just in case)

In [5]:
embedding_model = "msmarco-distilbert-cos-v5"
embedder = SentenceTransformer(embedding_model)

In [6]:
query = "coś w stylu harrego pottera, ale bardziej mroczne"
query_emb = embedder.encode_query(query)

corpus_embeddings = torch.from_numpy(
    np.vstack(data["embeddings"].values)
).float()

In [10]:
top_k = 5

similarity_scores = embedder.similarity(query_emb, corpus_embeddings)[0]
scores, indices = torch.topk(similarity_scores, k=top_k)

print("\nQuery:", query)
print("Top 5 most similar sentences in corpus:")

for score, idx in zip(scores, indices):
    print(f"(Score: {score:.4f})", data["summaries"].to_list()[idx], "\n")


Query: coś w stylu harrego pottera, ale bardziej mroczne
Top 5 most similar sentences in corpus:
(Score: 0.4111) Album zawiera materiały opublikowane pierwotnie w Chew #6-10. Detektyw cybopata Tony Chu, potrafiący odbierać mentalne odczucia ze wszystkiego, co zjada. 

(Score: 0.3843) Prípad rieši inšpektor Harry Hole, ktorý nedávno dostal tajomný list s podpisom Snehuliak. V Osle napadol prvý sneh. Jonas ráno vstane a zist 

(Score: 0.3779) Faust to dzieło życia Goethego, dramat o możliwościach ludzkiego poznania. Wielki uczony - wciąż spragniony wiedzy o sensie istnienia - zawiera 

(Score: 0.3342) J. Ryan Stradal's startlingly original debut tells the story of a single dish and character. By turns quirky, hilarious, and vividly sensory, Kitchens of the Great Midwest is an unexpected mother-daughter story. 

(Score: 0.3158) Filippo Brunelleschi's Dome is the story of how a Renaissance man bent men, materials, and the very forces of nature to build an architectural wonder. Over twenty